# Pilot Training — UNet++ EfficientNet-B0, revised from-scratch run

**Classes:**
- `0` = background (black)
- `1` = neuropil (yellow)
- `2` = soma (red)

**Main changes in this revised version:**
- Saves checkpoints by biologically useful metrics, not only validation loss: `best_foreground.pt`, `best_soma.pt`, `best_neuropil.pt`, `best_loss.pt`, `last.pt`, and every `epoch_XXX.pt`.
- Early stopping now follows `foreground_score = 0.3 * dice_neu + 0.7 * dice_soma` instead of `val_loss`.
- Reduces false-positive pressure by using mild soma oversampling (`target_soma_fraction=0.20`) and **unweighted** cross-entropy by default.
- Keeps optional mild manual class weights available, but does not use inverse-frequency weights by default.
- Simplifies soma augmentation by removing elastic/grid distortion and heavy color/shape transforms.
- Logs precision and recall for neuropil and soma so false positives vs false negatives can be diagnosed.
- Full-slide validation now also reports precision and recall.

Run this notebook **from scratch**. Do not run the old continue-training cell for this corrected pilot.



> **Repository version:** Notebook outputs were cleared and paths/split IDs replaced by placeholders. This notebook reproduces the *from-scratch* training workflow; to **fine-tune a pretrained checkpoint**, use `scripts/fine_tune.py`. See the repository README for details. The original notebook is not a runnable demo until you supply images, class-ID masks, and train/validation IDs.

## 1. Imports

In [ ]:
import numpy as np
import cv2
import os
import math
import random
import pandas as pd
import torch
import torch.nn as nn
import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import segmentation_models_pytorch as smp

from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from tqdm.notebook import tqdm

## 2. Paths and Dataset Splits

Define directory paths and the explicit list of image IDs for each split.
Using explicit ID lists (instead of scanning a directory) prevents accidental
leakage of validation or test images into the training set.

In [ ]:
# Configure your own image directories and image-level splits before running.
# Paths are relative to the repository root when Jupyter starts there.
PATH_MASKS = "data/masks_ids"       # 0/1/2 class-ID PNG masks
PATH_NORMALIZED = "data/images_norm" # pre-normalized RGB PNG images
PATH_SPLITS = "outputs/training"
OUT_DIR = "outputs/training/checkpoints"

os.makedirs(PATH_SPLITS, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Replace each entry with a real PNG basename (without .png).
# Keep tissue sections / animals from the same biological source out of both
# training and validation whenever feasible to reduce data leakage.
TRAIN_IDS = ["train_image_1", "train_image_2"]
VAL_IDS = ["val_image_1"]
TEST_IDS = []

BG_ID, NEU_ID, SOMA_ID = 0, 1, 2
NUM_CLASSES = 3
ID2RGB = {BG_ID: (0, 0, 0), NEU_ID: (255, 255, 0), SOMA_ID: (255, 0, 0)}


## 3. Device Setup

In [ ]:
# So that the code can be run on any platform, support apple silicon GPU (MPS), NVIDIA GPU (CUDA), or CPU.
# MPS = Apple Silicon GPU; falls back to CUDA (NVIDIA) or CPU.
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")

# confirm the device is functional
_ = torch.randn(512, 512, device=device)
print("Device is working.")

## 4. Preprocessing Utilities

These helpers operate on individual images:
- `compute_pad_to_multiple`: calculates symmetric padding so image dimensions are divisible by the patch size
- `generate_patch_grid`: returns top-left (x, y) corner coordinates for all overlapping patches
- `pad_with_known_pad`: applies precomputed padding (reflection for images, constant-0 for masks)

In [ ]:
def compute_pad_to_multiple(h, w, multiple=512):
    """
    Compute symmetric padding to make (H, W) evenly divisible by `multiple`.

    Parameters
    ----------
    h, w     : int  — original image height and width
    multiple : int  — target divisor (equals the patch size)

    Returns
    -------
    pad      : (top, bottom, left, right) ints
    new_size : (new_h, new_w) after padding
    """
    new_h = math.ceil(h / multiple) * multiple
    new_w = math.ceil(w / multiple) * multiple
    pad_h, pad_w = new_h - h, new_w - w
    top,  bottom = pad_h // 2, pad_h - pad_h // 2
    left, right  = pad_w // 2, pad_w - pad_w // 2
    return (top, bottom, left, right), (new_h, new_w)


def generate_patch_grid(h, w, patch_size=512, stride=256):
    """
    Generate all overlapping patch top-left coordinates for a padded image.

    A stride < patch_size produces overlapping patches, which improves
    boundary quality during inference (averaged logits in overlap zones).

    Parameters
    ----------
    h, w        : int — padded image dimensions (already divisible by patch_size)
    patch_size  : int — side length of each square patch
    stride      : int — step between consecutive patch origins (use patch_size//2 for 50% overlap)

    Returns
    -------
    list of (x, y) tuples — top-left corner of each patch
    """
    xs = list(range(0, w - patch_size + 1, stride))
    ys = list(range(0, h - patch_size + 1, stride))
    return [(x, y) for y in ys for x in xs]


def pad_with_known_pad(img, pad, is_mask=False, bg_value=0):
    """
    Apply a precomputed (top, bottom, left, right) border to an image or mask.

    Images use BORDER_CONSTANT to extend white background of slide padding 
    Masks use CONSTANT padding as well with bg_value=0 (background), so padded
    pixels never contribute to class loss.
    """
    top, bottom, left, right = pad
    if is_mask:
        border_type = cv2.BORDER_CONSTANT
        fill_value  = bg_value          # 0 = background class
    else:
        border_type = cv2.BORDER_CONSTANT
        fill_value  = 255               # white, matching slide mounting area
    return cv2.copyMakeBorder(img, top, bottom, left, right,
                              borderType=border_type, value=fill_value)


def unpad_np(arr, pad):
    """
    Remove padding from a 2-D or 3-D numpy array to restore original dimensions.

    Parameters
    ----------
    arr : np.ndarray  shape (H_pad, W_pad) or (H_pad, W_pad, C)
    pad : (top, bottom, left, right)
    """
    top, bottom, left, right = pad
    h, w = arr.shape[:2]
    return arr[top: h - bottom if bottom > 0 else h,
               left: w - right  if right  > 0 else w]

## 5. Patch Catalogue Builder

For each split (train / val), iterate over the explicit image ID list,
pad each mask, generate the patch grid, and record:
- which image the patch belongs to
- its top-left coordinate
- whether it contains any soma pixel (`has_soma`)
- how many soma pixels it contains (`soma_pixel_count`) --> used later for sampling weights

**Key change from single image pilot 1:** validation patches come from a
fully separate image (VAL_IDS), not a right-stripe of a training image.
This means val loss reflects true cross-image generalization.

In [ ]:
def build_patch_catalogue(
    image_ids,
    masks_dir,
    patch_size=512,
    stride=256,
    soma_id=SOMA_ID,
):
    """
    Build a patch catalogue (DataFrame) and a pad lookup dict for a list of images.

    Each row in the returned DataFrame describes one patch:
        image_id         : stem of the source image file
        x, y             : top-left pixel coordinate in the *padded* image space
        patch_size       : side length (constant, kept for reference)
        has_soma         : 1 if any soma pixel exists in this patch, else 0
        soma_pixel_count : exact number of soma pixels (finer weight than has_soma)

    Parameters
    ----------
    image_ids  : list[str]  — IDs of images to process (stems, no extension)
    masks_dir  : str        — directory containing <image_id>.png class-ID masks
    patch_size : int        — patch side length in pixels
    stride     : int        — stride between patch origins (patch_size//2 = 50% overlap)
    soma_id    : int        — class ID for soma class

    Returns
    -------
    df         : pd.DataFrame with patch-level info
    pad_lookup : dict  image_id -> (top, bottom, left, right)
    """
    rows = []
    pad_lookup = {}

    for image_id in image_ids:
        mask_path = os.path.join(masks_dir, f"{image_id}.png")
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)  # HxW uint8

        if mask is None:
            raise FileNotFoundError(f"Could not read mask: {mask_path}")
        if mask.ndim != 2:
            raise ValueError(f"Mask must be HxW; got shape {mask.shape} for {image_id}")

        h, w = mask.shape
        pad, (h_pad, w_pad) = compute_pad_to_multiple(h, w, multiple=patch_size)
        pad_lookup[image_id] = pad

        mask_pad = pad_with_known_pad(mask, pad, is_mask=True, bg_value=BG_ID)
        coords   = generate_patch_grid(h_pad, w_pad, patch_size=patch_size, stride=stride)

        for (x, y) in coords:
            patch            = mask_pad[y:y + patch_size, x:x + patch_size]
            soma_count       = int(np.sum(patch == soma_id))
            has_soma         = int(soma_count > 0)
            rows.append((image_id, x, y, patch_size, has_soma, soma_count))

        print(f"  {image_id}: original ({h}, {w}) -> padded ({h_pad}, {w_pad}) "
              f"| {len(coords)} patches "
              f"| soma patches: {sum(r[4] for r in rows if r[0] == image_id)}")

    cols = ["image_id", "x", "y", "patch_size", "has_soma", "soma_pixel_count"]
    df   = pd.DataFrame(rows, columns=cols)
    return df, pad_lookup


def save_pad_lookup(pad_lookup, path):
    """Serialize the pad_lookup dict to a CSV for later reuse in inference."""
    rows = [
        (iid, *pad)
        for iid, pad in pad_lookup.items()
    ]
    pd.DataFrame(rows, columns=["image_id","pad_top","pad_bottom","pad_left","pad_right"]
                 ).to_csv(path, index=False)


def load_pad_lookup(pad_csv):
    """Deserialize the pad_lookup dict from a saved CSV."""
    df = pd.read_csv(pad_csv).set_index("image_id")
    return {
        idx: (int(row["pad_top"]), int(row["pad_bottom"]),
              int(row["pad_left"]), int(row["pad_right"]))
        for idx, row in df.iterrows()
    }

In [ ]:
# --------------------------------------------------------------------------
# Run preprocessing for TRAIN and VAL splits separately
# --------------------------------------------------------------------------
PATCH_SIZE = 512
STRIDE     = 256  # 50% overlap; reduces tile-boundary artifacts during inference

print("Building TRAIN catalogue")
train_df, train_pad_lookup = build_patch_catalogue(
    image_ids  = TRAIN_IDS,
    masks_dir  = PATH_MASKS,
    patch_size = PATCH_SIZE,
    stride     = STRIDE,
)

print("\nBuilding VAL catalogue")
val_df, val_pad_lookup = build_patch_catalogue(
    image_ids  = VAL_IDS,
    masks_dir  = PATH_MASKS,
    patch_size = PATCH_SIZE,
    stride     = STRIDE,
)

# Merge pad lookups
all_pad_lookup = {**train_pad_lookup, **val_pad_lookup}

# Save CSVs
train_df.to_csv(os.path.join(PATH_SPLITS, "train_patches.csv"), index=False)
val_df.to_csv(  os.path.join(PATH_SPLITS, "val_patches.csv"),   index=False)
save_pad_lookup(all_pad_lookup, os.path.join(PATH_SPLITS, "pad_info.csv"))

print(f"\nTrain patches : {len(train_df):,}  |  soma fraction: {train_df['has_soma'].mean():.3f}")
print(f"Val   patches : {len(val_df):,}  |  soma fraction: {val_df['has_soma'].mean():.3f}")

## 6. Weighted Sampler — Mild Soma Oversampling

The previous run used stronger class-imbalance pressure: soma oversampling plus inverse-frequency class weights. Because visual inspection showed many false positives, this version reduces that pressure.

Default for this revised run:
- sample soma-containing patches at a mild target fraction of `0.20`
- keep `pixel_boost=False`
- use unweighted `CrossEntropyLoss` later

If soma recall becomes too low, increase `target_soma_fraction` to `0.25` before adding class weights back.


In [ ]:
def build_weighted_sampler(df, target_soma_fraction=0.20, pixel_boost=False):
    """
    Build a WeightedRandomSampler that mildly oversamples soma-containing patches.

    This version is intentionally conservative because the prior run produced
    many soma/neuropil false positives. We want the model to see rare soma
    patches often enough to learn them, but not so often that it hallucinates
    foreground across the whole slide.

    Parameters
    ----------
    df                   : DataFrame with 'has_soma' and 'soma_pixel_count'
    target_soma_fraction : desired fraction of soma patches per epoch
                           Recommended first-pass value: 0.20
    pixel_boost          : if True, further boosts patches with many soma pixels.
                           Keep False for this pilot.
    """
    p = float(df["has_soma"].mean())

    if p <= 0:
        raise ValueError("No soma-containing patches found in train_df. Check masks/catalogue.")

    soma_weight = (target_soma_fraction * (1 - p)) / (p * (1 - target_soma_fraction) + 1e-8)
    soma_weight = float(np.clip(soma_weight, 1.0, 100.0))

    weights = np.where(df["has_soma"].to_numpy() == 1, soma_weight, 1.0).astype(np.float32)

    if pixel_boost:
        pixel_log = np.log1p(df["soma_pixel_count"].to_numpy().astype(np.float32))
        soma_mask = df["has_soma"].to_numpy() == 1
        if soma_mask.any():
            pixel_log_norm = pixel_log / (pixel_log[soma_mask].max() + 1e-8)
            weights[soma_mask] *= (1.0 + pixel_log_norm[soma_mask])

    weights_t = torch.tensor(weights, dtype=torch.float32)
    sampler   = WeightedRandomSampler(weights=weights_t, num_samples=len(df), replacement=True)

    expected_soma_frac = float(weights[df["has_soma"].to_numpy() == 1].sum() / weights.sum())

    print(f"  Base p(soma)              : {p:.3f}")
    print(f"  Target p(soma) in sampler : {target_soma_fraction:.3f}")
    print(f"  Expected p(soma) sampled  : {expected_soma_frac:.3f}")
    print(f"  soma_weight               : {soma_weight:.2f}")
    print(f"  pixel_boost               : {pixel_boost}")
    return sampler


print("Building training sampler...")
sampler = build_weighted_sampler(train_df, target_soma_fraction=0.20, pixel_boost=False)


## 7. Data Augmentation

Two augmentation profiles are defined:

- **Weak aug** (soma-free patches): horizontal/vertical flips, mild rotation, gentle brightness/contrast jitter. Teaches basic spatial invariance without distorting background texture too aggressively.
- **Strong aug** (soma-containing patches): all of the above plus elastic deformation, grid distortion, coarser geometric transforms, stronger color jitter, and noise. The goal is to artificially multiply the effective number of soma examples the model sees.

In [ ]:
def make_weak_aug():
    """
    Light augmentation for all patches.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
        A.RandomGamma(gamma_limit=(90, 110), p=0.2),
    ])


def make_strong_aug():
    """
    Moderate augmentation for soma-containing patches.

    IMPORTANT CHANGE:
    The previous run used ElasticTransform, GridDistortion, stronger hue shifts,
    blur, and sharpening. Because you saw many false positives and unstable
    soma learning, this version removes those aggressive transforms for now.
    We keep only realistic flips/rotations, mild affine jitter, and mild intensity
    variation.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.ShiftScaleRotate(
            shift_limit=0.03,
            scale_limit=0.05,
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.4,
        ),

        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.5),
        A.RandomGamma(gamma_limit=(85, 120), p=0.3),
        A.GaussNoise(p=0.15),
    ])


## 8. Dataset Class

`PatchDataset` loads full images on demand (with an LRU cache to avoid
re-reading from disk every iteration), applies precomputed padding, slices
the requested patch, and runs conditional augmentation.

**Changes vs. single-image pilot:**
- `cache_size` is set to `len(image_ids)` by default so all training images
  stay in memory simultaneously (beware that you must have a decent RAM to run this).
- Padding is applied *inside* `__getitem__` using the pre-built `pad_lookup`
  so the raw (unpadded) images are what get cached (saves memory).
- `soma_id` attribute is referenced from constants, not hard-coded.

In [ ]:
class PatchDataset(Dataset):
    """
    PyTorch Dataset that serves 512x512 patches from multiple whole-slide images.

    Workflow per item:
      1. Look up the image_id + patch coordinates from the pre-built CSV catalogue.
      2. Load the full image + mask from disk (or from cache if recently accessed).
      3. Apply precomputed padding.
      4. Slice the patch.
      5. Apply conditional augmentation (strong for soma patches, weak otherwise).
      6. Convert to float32 tensor (image) and int64 tensor (mask).

    Parameters
    ----------
    df          : pd.DataFrame  — patch catalogue (from build_patch_catalogue)
    images_dir  : str           — directory with normalized RGB PNGs
    masks_dir   : str           — directory with class-ID PNGs (only for training/val)
    pad_lookup  : dict          — image_id -> (top,bottom,left,right)
    patch_size  : int           — patch side length
    weak_aug    : A.Compose     — augmentation for non-soma patches (pass None for val)
    strong_aug  : A.Compose     — augmentation for soma patches    (pass None for val)
    cache_size  : int           — max number of images to keep in RAM simultaneously
    has_masks   : bool          — False for unlabeled test images (skips mask loading)
    """

    def __init__(
        self,
        df,
        images_dir,
        masks_dir,
        pad_lookup,
        patch_size = 512,
        weak_aug   = None,
        strong_aug = None,
        cache_size = 5,
        has_masks  = True,
    ):
        self.df          = df.reset_index(drop=True)
        self.images_dir  = images_dir
        self.masks_dir   = masks_dir
        self.pad_lookup  = pad_lookup
        self.patch_size  = patch_size
        self.weak_aug    = weak_aug
        self.strong_aug  = strong_aug
        self.has_masks   = has_masks
        self.soma_id     = SOMA_ID

        # LRU image cache: stores (img_rgb, mask_ids) tuples keyed by image_id
        # Keeping all training images in RAM avoids repeated disk I/O during
        # an epoch. Reduce cache_size if you hit memory limits.
        self.cache_size  = cache_size
        self._cache      = {}
        self._cache_order = []

    # ------------------------------------------------------------------
    # Internal cache helpers
    # ------------------------------------------------------------------
    def _cache_get(self, image_id):
        if image_id in self._cache:
            self._cache_order.remove(image_id)
            self._cache_order.append(image_id)
            return self._cache[image_id]
        return None

    def _cache_put(self, image_id, value):
        if image_id in self._cache:
            self._cache_order.remove(image_id)
        self._cache[image_id] = value
        self._cache_order.append(image_id)
        while len(self._cache_order) > self.cache_size:
            evict = self._cache_order.pop(0)
            self._cache.pop(evict, None)

    # ------------------------------------------------------------------
    # Image + mask loader
    # ------------------------------------------------------------------
    def _load_full(self, image_id):
        """
        Load raw (un-padded) image and optional mask from disk.
        Returns (img_rgb [HxWx3 uint8], mask_ids [HxW uint8] or None)
        """
        cached = self._cache_get(image_id)
        if cached is not None:
            return cached

        img_path = os.path.join(self.images_dir, f"{image_id}.png")
        img_bgr  = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        mask_ids = None
        if self.has_masks:
            mask_path = os.path.join(self.masks_dir, f"{image_id}.png")
            mask_ids  = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
            if mask_ids is None:
                raise FileNotFoundError(f"Mask not found: {mask_path}")
            if mask_ids.ndim != 2:
                raise ValueError(f"Mask must be HxW; got {mask_ids.shape}")

        self._cache_put(image_id, (img_rgb, mask_ids))
        return img_rgb, mask_ids

    # ------------------------------------------------------------------
    # Dataset interface
    # ------------------------------------------------------------------
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        image_id = row["image_id"]
        x, y     = int(row["x"]), int(row["y"])
        ps       = int(row["patch_size"])
        has_soma = int(row["has_soma"])

        img_rgb, mask_ids = self._load_full(image_id)
        pad = self.pad_lookup[image_id]

        # Apply padding to the full image/mask before slicing the patch
        img_pad  = pad_with_known_pad(img_rgb, pad, is_mask=False)
        img_patch = img_pad[y:y + ps, x:x + ps]

        if self.has_masks:
            mask_pad  = pad_with_known_pad(mask_ids, pad, is_mask=True, bg_value=BG_ID)
            mask_patch = mask_pad[y:y + ps, x:x + ps]
        else:
            # Dummy mask for inference-only mode (not used in loss calculation)
            mask_patch = np.zeros((ps, ps), dtype=np.uint8)

        # Conditional augmentation:
        # Soma patches get the heavier transform to synthesize more variety.
        # Non-soma patches get the lighter transform (less risk of distorting
        # the already-well-represented neuropil texture).
        aug = self.strong_aug if (has_soma == 1 and self.strong_aug is not None) \
              else self.weak_aug

        if aug is not None:
            out        = aug(image=img_patch, mask=mask_patch)
            img_patch  = out["image"]
            mask_patch = out["mask"]

        # img : (H, W, C) uint8 -> (C, H, W) float32 in [0, 1]
        img_tensor  = torch.from_numpy(img_patch).permute(2, 0, 1).float() / 255.0
        # mask: (H, W) uint8 -> (H, W) int64 (required by CrossEntropyLoss)
        mask_tensor = torch.from_numpy(mask_patch.copy()).long()

        return img_tensor, mask_tensor, has_soma

## 9. Class Weights: compute for reference, but do not use by default

The old run used inverse-frequency class weights together with soma oversampling. That likely made foreground false positives worse.

This cell still computes class counts and two possible manual weight vectors for reference, but the model/loss cell below uses **unweighted** cross-entropy by default.


In [ ]:
def compute_class_counts_from_masks(image_ids, masks_dir, num_classes=NUM_CLASSES):
    """Count pixels per class in the training masks."""
    counts = np.zeros(num_classes, dtype=np.float64)

    for image_id in image_ids:
        mask_path = os.path.join(masks_dir, f"{image_id}.png")
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        if mask is None:
            print(f"WARNING: mask not found: {mask_path}")
            continue
        for c in range(num_classes):
            counts[c] += np.sum(mask == c)

    print("Class pixel counts:", dict(zip(["bg", "neuropil", "soma"], counts.astype(int))))
    print("Class pixel fractions:", dict(zip(["bg", "neuropil", "soma"], np.round(counts / counts.sum(), 6))))
    return counts


class_counts = compute_class_counts_from_masks(TRAIN_IDS, PATH_MASKS)

# Default: no class weights. This is intentionally less aggressive.
LOSS_WEIGHTS_MODE = "none"  # options: "none", "mild_manual"

if LOSS_WEIGHTS_MODE == "none":
    class_weights = None
    print("Using unweighted CrossEntropyLoss.")
elif LOSS_WEIGHTS_MODE == "mild_manual":
    # Use only if soma recall is too low. Avoid inverse-frequency weights for now.
    class_weights = torch.tensor([0.2, 1.0, 2.0], dtype=torch.float32)
    print("Using mild manual class weights:", class_weights.tolist())
else:
    raise ValueError(f"Unknown LOSS_WEIGHTS_MODE: {LOSS_WEIGHTS_MODE}")


## 10. Build Datasets and DataLoaders

In [ ]:
weak_aug   = make_weak_aug()
strong_aug = make_strong_aug()

# Training dataset: augmentation on, cache all 5 training images
train_dataset = PatchDataset(
    df         = train_df,
    images_dir = PATH_NORMALIZED,
    masks_dir  = PATH_MASKS,
    pad_lookup = all_pad_lookup,
    patch_size = PATCH_SIZE,
    weak_aug   = weak_aug,
    strong_aug = strong_aug,
    cache_size = len(TRAIN_IDS),   # keep all training images in RAM
    has_masks  = True,
)

# Validation dataset: NO augmentation — we need a stable, reproducible metric
val_dataset = PatchDataset(
    df         = val_df,
    images_dir = PATH_NORMALIZED,
    masks_dir  = PATH_MASKS,
    pad_lookup = all_pad_lookup,
    patch_size = PATCH_SIZE,
    weak_aug   = None,
    strong_aug = None,
    cache_size = len(VAL_IDS),
    has_masks  = True,
)

# Batch size: increase if VRAM allows; start at 4 and monitor GPU memory
BATCH_SIZE  = 4
NUM_WORKERS = 0   # set 0 on macOS MPS; safe default to avoid multiprocessing issues

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,     # WeightedRandomSampler takes over shuffle logic
    num_workers = NUM_WORKERS,
    pin_memory  = False,
    drop_last   = True,        # avoid 1-sample batches that can crash BatchNorm
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = False,
)

# Quick sanity check
imgs, masks, has_soma = next(iter(train_loader))
print(f"Batch imgs  : {imgs.shape}  dtype={imgs.dtype}  range=[{imgs.min():.2f}, {imgs.max():.2f}]")
print(f"Batch masks : {masks.shape}  dtype={masks.dtype}  unique={torch.unique(masks).tolist()}")
print(f"has_soma    : {has_soma.tolist()}")

## 11. Verify Sampler Effectiveness

Sample 200 batches and measure the empirical soma fraction.
It should be close to the `target_soma_fraction` set in the sampler.

In [ ]:
N_CHECK = 200
soma_count_check = 0
for i, (_, _, hs) in zip(range(N_CHECK), train_loader):
    soma_count_check += int(hs.sum().item())
total_patches_checked = N_CHECK * BATCH_SIZE
print(f"Empirical soma fraction over {total_patches_checked} patches: "
      f"{soma_count_check / total_patches_checked:.3f}  (target: 0.30)")

## 12. Model, Loss, Optimizer, Scheduler

**Default loss for this corrected run:** unweighted `CrossEntropyLoss`.

This is intentional. We are already using mild soma oversampling. Adding inverse-frequency class weights made the rare foreground classes too expensive to miss and may have encouraged false positives.

If the new run becomes too conservative and misses soma, first try `target_soma_fraction=0.25`. If that is still insufficient, set `LOSS_WEIGHTS_MODE = "mild_manual"` in the previous cell and rerun from scratch.


In [ ]:
# --------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------
model = smp.UnetPlusPlus(
    encoder_name    = "efficientnet-b0",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = NUM_CLASSES,
)
model = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# --------------------------------------------------------------------------
# Loss
# --------------------------------------------------------------------------
if class_weights is None:
    criterion = nn.CrossEntropyLoss()
    print("Loss: unweighted CrossEntropyLoss")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    print("Loss: mildly weighted CrossEntropyLoss", class_weights.tolist())

# --------------------------------------------------------------------------
# Optimizer
# --------------------------------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# --------------------------------------------------------------------------
# LR Scheduler
# Keep scheduler on val_loss because it is smooth, but checkpointing/early
# stopping below uses foreground_score.
# --------------------------------------------------------------------------
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode     = "min",
    factor   = 0.5,
    patience = 3,
    min_lr   = 1e-6
)


## 13. Training and Validation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Run one full pass over the training DataLoader."""
    model.train()
    total_loss, n = 0.0, 0

    for imgs, masks, _ in tqdm(loader, desc="Train", leave=False):
        imgs  = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss   = criterion(logits, masks)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        bs          = imgs.size(0)
        total_loss += loss.item() * bs
        n          += bs

    return total_loss / max(n, 1)


@torch.no_grad()
def confusion_matrix_torch(preds, masks, num_classes=NUM_CLASSES):
    """Fast confusion matrix for one batch. Rows=true class, cols=pred class."""
    preds = preds.view(-1).to(torch.int64)
    masks = masks.view(-1).to(torch.int64)
    valid = (masks >= 0) & (masks < num_classes)
    inds = num_classes * masks[valid] + preds[valid]
    cm = torch.bincount(inds, minlength=num_classes ** 2).reshape(num_classes, num_classes)
    return cm.cpu().numpy().astype(np.float64)


def metrics_from_cm(cm, eps=1e-7):
    """
    Return Dice, IoU, precision, and recall from a confusion matrix.
    Rows=true class, cols=pred class.
    """
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    dice      = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou       = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall    = (tp + eps) / (tp + fn + eps)
    return dice, iou, precision, recall


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """
    Run one full pass over the validation DataLoader.

    Returns a dictionary so we can log Dice, precision, recall, and the
    foreground score used for checkpointing.
    """
    model.eval()
    total_loss = 0.0
    n          = 0
    cm_total   = np.zeros((num_classes, num_classes), dtype=np.float64)

    for imgs, masks, _ in tqdm(loader, desc="Val", leave=False):
        imgs  = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)
        loss   = criterion(logits, masks)
        preds  = torch.argmax(logits, dim=1)

        bs          = imgs.size(0)
        total_loss += loss.item() * bs
        n          += bs
        cm_total   += confusion_matrix_torch(preds, masks, num_classes)

    dice, iou, precision, recall = metrics_from_cm(cm_total)
    val_loss = total_loss / max(n, 1)

    # Weighted toward soma because soma detection is the biologically critical problem.
    foreground_score = 0.3 * dice[NEU_ID] + 0.7 * dice[SOMA_ID]

    return {
        "val_loss"         : float(val_loss),
        "dice"             : dice.tolist(),
        "iou"              : iou.tolist(),
        "precision"        : precision.tolist(),
        "recall"           : recall.tolist(),
        "foreground_score" : float(foreground_score),
        "confusion_matrix" : cm_total.astype(int).tolist(),
    }


## 14. Training Loop

This loop now saves multiple checkpoints:

- `last.pt`: latest epoch, useful for resuming after interruption
- `epoch_XXX.pt`: every epoch, useful for visual comparison later
- `best_loss.pt`: lowest validation loss
- `best_soma.pt`: highest validation soma Dice
- `best_neuropil.pt`: highest validation neuropil Dice
- `best_foreground.pt`: highest `0.3 * dice_neu + 0.7 * dice_soma`

For this project, use `best_foreground.pt` for the main qualitative/full-slide evaluation, not `best_loss.pt`.


In [ ]:
# --------------------------------------------------------------------------
# Training configuration for a clean from-scratch run
# --------------------------------------------------------------------------
EPOCHS         = 100
PATIENCE       = 20      # foreground-score patience, not val-loss patience
MIN_DELTA      = 1e-4
CHECKPOINT_DIR = OUT_DIR
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_loss          = float("inf")
best_soma_dice         = -float("inf")
best_neuropil_dice     = -float("inf")
best_foreground_score  = -float("inf")
patience_counter       = 0
history                = []

# Save run settings for reproducibility
run_settings = {
    "patch_size": PATCH_SIZE,
    "stride": STRIDE,
    "batch_size": BATCH_SIZE,
    "epochs_max": EPOCHS,
    "early_stopping_metric": "foreground_score = 0.3*dice_neu + 0.7*dice_soma",
    "target_soma_fraction": 0.20,
    "loss_weights_mode": LOSS_WEIGHTS_MODE,
    "optimizer": "AdamW(lr=1e-4, weight_decay=1e-4)",
    "encoder": "efficientnet-b0 imagenet",
}
pd.Series(run_settings).to_csv(os.path.join(PATH_SPLITS, "run_settings.csv"))

for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = validate_one_epoch(model, val_loader, criterion, device)

    val_loss         = val_metrics["val_loss"]
    val_dice         = val_metrics["dice"]
    val_precision    = val_metrics["precision"]
    val_recall       = val_metrics["recall"]
    foreground_score = val_metrics["foreground_score"]

    # Scheduler still watches val_loss because it is smoother than Dice.
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    # Save latest and every-epoch checkpoints.
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "last.pt"))
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"epoch_{epoch:03d}.pt"))

    improved_loss = val_loss < best_val_loss - MIN_DELTA
    if improved_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_loss.pt"))

    improved_soma = val_dice[SOMA_ID] > best_soma_dice + MIN_DELTA
    if improved_soma:
        best_soma_dice = val_dice[SOMA_ID]
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_soma.pt"))

    improved_neu = val_dice[NEU_ID] > best_neuropil_dice + MIN_DELTA
    if improved_neu:
        best_neuropil_dice = val_dice[NEU_ID]
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_neuropil.pt"))

    improved_foreground = foreground_score > best_foreground_score + MIN_DELTA
    if improved_foreground:
        best_foreground_score = foreground_score
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_foreground.pt"))
    else:
        patience_counter += 1

    history.append({
        "epoch"              : epoch,
        "train_loss"         : train_loss,
        "val_loss"           : val_loss,
        "foreground_score"   : foreground_score,
        "dice_bg"            : val_dice[BG_ID],
        "dice_neu"           : val_dice[NEU_ID],
        "dice_soma"          : val_dice[SOMA_ID],
        "precision_bg"       : val_precision[BG_ID],
        "precision_neu"      : val_precision[NEU_ID],
        "precision_soma"     : val_precision[SOMA_ID],
        "recall_bg"          : val_recall[BG_ID],
        "recall_neu"         : val_recall[NEU_ID],
        "recall_soma"        : val_recall[SOMA_ID],
        "lr"                 : current_lr,
        "best_loss_model"    : improved_loss,
        "best_soma_model"    : improved_soma,
        "best_neuropil_model": improved_neu,
        "best_foreground_model": improved_foreground,
        "patience"           : patience_counter,
    })

    markers = []
    if improved_loss:       markers.append("best_loss")
    if improved_soma:       markers.append("best_soma")
    if improved_neu:        markers.append("best_neu")
    if improved_foreground: markers.append("best_fg")
    marker_text = " | " + ", ".join(markers) if markers else ""

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train={train_loss:.4f} | val={val_loss:.4f} | fg={foreground_score:.3f} | "
        f"Dice bg={val_dice[BG_ID]:.3f} neu={val_dice[NEU_ID]:.3f} soma={val_dice[SOMA_ID]:.3f} | "
        f"Prec neu={val_precision[NEU_ID]:.3f} soma={val_precision[SOMA_ID]:.3f} | "
        f"Rec neu={val_recall[NEU_ID]:.3f} soma={val_recall[SOMA_ID]:.3f} | "
        f"lr={current_lr:.2e} | patience={patience_counter}/{PATIENCE}"
        f"{marker_text}"
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(PATH_SPLITS, "training_history.csv"), index=False)

    if patience_counter >= PATIENCE:
        print(
            f"\nEarly stopping triggered at epoch {epoch}. "
            f"Best foreground_score: {best_foreground_score:.4f}"
        )
        break

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(PATH_SPLITS, "training_history.csv"), index=False)
print("\nTraining complete.")
print("Best val loss        :", best_val_loss)
print("Best soma Dice       :", best_soma_dice)
print("Best neuropil Dice   :", best_neuropil_dice)
print("Best foreground score:", best_foreground_score)
print("Main checkpoint to inspect: best_foreground.pt")


In [ ]:
# --------------------------------------------------------------------------
# OPTIONAL: compare selected checkpoints after the from-scratch run
# --------------------------------------------------------------------------
# Do not run the old continue-training workflow for this corrected pilot.
# After training finishes, use this list to run qualitative/full-slide inference
# on multiple checkpoints if you want to visually inspect learning over time.

CHECKPOINTS_TO_COMPARE = [
    "best_foreground.pt",
    "best_soma.pt",
    "best_neuropil.pt",
    "best_loss.pt",
    "last.pt",
]
 
# You can also manually add specific epochs after looking at training_history.csv:
# CHECKPOINTS_TO_COMPARE += ["epoch_003.pt", "epoch_005.pt", "epoch_010.pt"]

print("Available comparison checkpoints:")
for ckpt_name in CHECKPOINTS_TO_COMPARE:
    ckpt_path = os.path.join(CHECKPOINT_DIR, ckpt_name)
    print(f"  {ckpt_name}: {'FOUND' if os.path.exists(ckpt_path) else 'missing'}")


## 15. Training Curves

Visual inspection of loss and Dice per class over epochs.

- Train and val loss should both decrease and converge.
- A large gap (train much lower than val) signals overfitting — increase augmentation or reduce model capacity.
- `dice_soma` starting low and gradually rising is expected and healthy.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
ax1.plot(history_df["epoch"], history_df["val_loss"],   label="Val loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Foreground/Dice curves
ax2.plot(history_df["epoch"], history_df["dice_bg"],   label="Dice bg")
ax2.plot(history_df["epoch"], history_df["dice_neu"],  label="Dice neuropil")
ax2.plot(history_df["epoch"], history_df["dice_soma"], label="Dice soma")
ax2.plot(history_df["epoch"], history_df["foreground_score"], label="Foreground score", linestyle="--")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Metric")
ax2.set_title("Validation Dice / Foreground Score")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PATH_SPLITS, "training_curves.png"), dpi=150)
plt.show()

# Precision/recall plot for diagnosing false positives vs false negatives
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(history_df["epoch"], history_df["precision_neu"],  label="Precision neuropil")
ax.plot(history_df["epoch"], history_df["recall_neu"],     label="Recall neuropil")
ax.plot(history_df["epoch"], history_df["precision_soma"], label="Precision soma")
ax.plot(history_df["epoch"], history_df["recall_soma"],    label="Recall soma")
ax.set_xlabel("Epoch")
ax.set_ylabel("Metric")
ax.set_title("Validation Precision/Recall")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PATH_SPLITS, "precision_recall_curves.png"), dpi=150)
plt.show()


## 16. Visualization Utilities

In [ ]:
def ids_to_rgb(mask_ids):
    """
    Convert a (H, W) integer class-ID mask to an (H, W, 3) RGB image
    using the ID2RGB colormap defined in Section 2.
    """
    out = np.zeros((*mask_ids.shape, 3), dtype=np.uint8)
    for k, rgb in ID2RGB.items():
        out[mask_ids == k] = rgb
    return out


def overlay_pred_on_image(img_rgb, mask_ids, alpha=0.35):
    """
    Blend the color-coded prediction mask onto the original image.
    alpha controls mask opacity (0 = invisible, 1 = opaque).
    """
    mask_rgb = ids_to_rgb(mask_ids).astype(np.float32)
    out = (1 - alpha) * img_rgb.astype(np.float32) + alpha * mask_rgb
    return np.clip(out, 0, 255).astype(np.uint8)


def class_legend():
    """Return matplotlib legend patches for the three segmentation classes."""
    return [
        mpatches.Patch(color=np.array(ID2RGB[BG_ID])   / 255, label="Background"),
        mpatches.Patch(color=np.array(ID2RGB[NEU_ID])  / 255, label="Neuropil"),
        mpatches.Patch(color=np.array(ID2RGB[SOMA_ID]) / 255, label="Soma"),
    ]

## 17. Patch-Level Qualitative Check (Validation Image)

Show a sample of validation patches: raw image, ground-truth mask,
predicted mask, and overlay. Run after loading the best checkpoint.

In [ ]:
# Load the checkpoint that is most aligned with your biological goal.
# Default: best_foreground.pt, not best_loss.pt.
CHECKPOINT_TO_LOAD = "best_foreground.pt"

model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, CHECKPOINT_TO_LOAD), map_location="cpu"))
model = model.to(device)
model.eval()
print(f"Loaded {CHECKPOINT_TO_LOAD}")


@torch.no_grad()
def show_val_patches(model, val_loader, device, n_soma=4, n_nosoma=2):
    """
    Display a curated sample of validation patches with GT and predictions.
    """
    model.eval()
    soma_shown, nosoma_shown = 0, 0

    for imgs, masks, has_soma in val_loader:
        if soma_shown >= n_soma and nosoma_shown >= n_nosoma:
            break

        imgs  = imgs.to(device)
        logits = model(imgs)
        preds  = torch.argmax(logits, dim=1).detach().cpu().numpy().astype(np.uint8)

        imgs_np  = (imgs.detach().cpu().permute(0,2,3,1).numpy() * 255).astype(np.uint8)
        masks_np = masks.numpy().astype(np.uint8)
        hs_np    = has_soma.numpy()

        for i in range(imgs_np.shape[0]):
            hs = int(hs_np[i])
            if hs == 1 and soma_shown >= n_soma:
                continue
            if hs == 0 and nosoma_shown >= n_nosoma:
                continue

            img, gt, pr = imgs_np[i], masks_np[i], preds[i]
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))

            axes[0].imshow(img);                   axes[0].set_title(f"Image  (has_soma={hs})")
            axes[1].imshow(ids_to_rgb(gt));        axes[1].set_title(f"GT     uniq={np.unique(gt).tolist()}")
            axes[2].imshow(ids_to_rgb(pr));        axes[2].set_title(f"Pred   uniq={np.unique(pr).tolist()}")
            axes[3].imshow(overlay_pred_on_image(img, pr))
            axes[3].set_title("Overlay (pred)")

            for ax in axes:
                ax.axis("off")
            fig.legend(handles=class_legend(), loc="lower center", ncol=3, fontsize=9)
            plt.tight_layout()
            plt.show()

            if hs == 1: soma_shown   += 1
            else:       nosoma_shown += 1


show_val_patches(model, val_loader, device, n_soma=4, n_nosoma=2)


## 18. Full-Slide Inference Utilities

Tile the full (padded) image, run the model on each batch of tiles,
accumulate logits in the padded logit canvas (averaging overlapping zones),
take argmax, and unpad back to original dimensions.

Two inference modes:
- **With GT mask** (`run_full_slide_with_gt`): for labeled val images — computes Dice/IoU
- **Without GT mask** (`run_full_slide_no_gt`): for unlabeled test images — qualitative only

In [ ]:
@torch.no_grad()
def predict_full_slide(model, img_rgb, pad, device,
                       patch_size=512, stride=256,
                       num_classes=NUM_CLASSES, infer_batch_size=4):
    """
    Tile-predict a full slide image and return per-pixel class predictions.

    Algorithm:
      1. Pad the image to make dimensions divisible by patch_size.
      2. Generate overlapping patch grid (stride < patch_size).
      3. Forward each batch of tiles through the model.
      4. Accumulate raw logits in a full-resolution canvas.
      5. Average overlapping logits (reduces tile-boundary artifacts).
      6. Argmax -> class IDs on the padded canvas.
      7. Unpad back to original image dimensions.

    Parameters
    ----------
    img_rgb         : np.ndarray (H, W, 3) uint8 — original normalized image
    pad             : (top, bottom, left, right) precomputed padding
    infer_batch_size: int — tiles per forward pass; increase if VRAM allows

    Returns
    -------
    pred_ids : np.ndarray (H, W) uint8 — predicted class IDs in original space
    """
    model.eval()

    img_pad      = pad_with_known_pad(img_rgb, pad, is_mask=False)
    h_pad, w_pad = img_pad.shape[:2]
    coords       = generate_patch_grid(h_pad, w_pad, patch_size=patch_size, stride=stride)

    # Canvas for logit accumulation (float32 for averaging)
    logits_sum = np.zeros((num_classes, h_pad, w_pad), dtype=np.float32)
    count_sum  = np.zeros((h_pad, w_pad),              dtype=np.float32)

    for i in range(0, len(coords), infer_batch_size):
        batch_coords = coords[i:i + infer_batch_size]
        patches = [img_pad[y:y + patch_size, x:x + patch_size] for (x, y) in batch_coords]

        batch_t = (torch.from_numpy(np.stack(patches, axis=0))
                   .permute(0, 3, 1, 2).float() / 255.0).to(device)

        logits = model(batch_t).detach().cpu().numpy().astype(np.float32)

        for b, (x, y) in enumerate(batch_coords):
            logits_sum[:, y:y + patch_size, x:x + patch_size] += logits[b]
            count_sum[y:y + patch_size, x:x + patch_size]     += 1.0

    logits_avg = logits_sum / np.maximum(count_sum[np.newaxis, ...], 1e-6)
    pred_pad   = np.argmax(logits_avg, axis=0).astype(np.uint8)
    return unpad_np(pred_pad, pad)


def confusion_matrix_np(gt_ids, pred_ids, num_classes=NUM_CLASSES):
    """
    Compute a (C, C) confusion matrix where rows=true, cols=predicted.
    """
    gt   = gt_ids.ravel().astype(np.int64)
    pr   = pred_ids.ravel().astype(np.int64)
    cm   = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t in range(num_classes):
        for p in range(num_classes):
            cm[t, p] = np.sum((gt == t) & (pr == p))
    return cm


def dice_iou_from_cm(cm, eps=1e-7):
    """
    Compute per-class Dice and IoU from a confusion matrix.
    """
    cm  = cm.astype(np.float64)
    tp  = np.diag(cm)
    fp  = cm.sum(axis=0) - tp
    fn  = cm.sum(axis=1) - tp
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou  = (tp + eps) / (tp + fp + fn + eps)
    return dice, iou

## 19. Full-Slide Evaluation on Labeled Validation Image(s)

In [ ]:
def run_full_slide_with_gt(
    image_id, model, device,
    path_normalized, path_masks,
    pad_lookup, out_dir,
    patch_size=512, stride=256, infer_batch_size=4,
):
    """
    Run full-slide inference on a labeled image and compute quantitative metrics.

    Saves:
      <image_id>_pred_ids.png
      <image_id>_pred_rgb.png
      <image_id>_overlay.png
    """
    img_bgr  = cv2.imread(os.path.join(path_normalized, f"{image_id}.png"), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f"Image not found: {image_id}.png in {path_normalized}")
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    gt_ids = cv2.imread(os.path.join(path_masks, f"{image_id}.png"), cv2.IMREAD_UNCHANGED)
    if gt_ids is None:
        raise FileNotFoundError(f"Mask not found: {image_id}.png in {path_masks}")

    pad      = pad_lookup[image_id]
    pred_ids = predict_full_slide(model, img_rgb, pad, device,
                                  patch_size=patch_size, stride=stride,
                                  infer_batch_size=infer_batch_size)

    cm = confusion_matrix_np(gt_ids, pred_ids)
    dice, iou, precision, recall = metrics_from_cm(cm)

    os.makedirs(out_dir, exist_ok=True)
    pred_rgb = ids_to_rgb(pred_ids)
    overlay  = overlay_pred_on_image(img_rgb, pred_ids)

    cv2.imwrite(os.path.join(out_dir, f"{image_id}_pred_ids.png"), pred_ids)
    cv2.imwrite(os.path.join(out_dir, f"{image_id}_pred_rgb.png"), cv2.cvtColor(pred_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(out_dir, f"{image_id}_overlay.png"),  cv2.cvtColor(overlay,  cv2.COLOR_RGB2BGR))

    return {
        "image_id"            : image_id,
        "checkpoint"          : CHECKPOINT_TO_LOAD,
        "dice_bg"             : float(dice[BG_ID]),
        "dice_neuropil"       : float(dice[NEU_ID]),
        "dice_soma"           : float(dice[SOMA_ID]),
        "iou_bg"              : float(iou[BG_ID]),
        "iou_neuropil"        : float(iou[NEU_ID]),
        "iou_soma"            : float(iou[SOMA_ID]),
        "precision_bg"        : float(precision[BG_ID]),
        "precision_neuropil"  : float(precision[NEU_ID]),
        "precision_soma"      : float(precision[SOMA_ID]),
        "recall_bg"           : float(recall[BG_ID]),
        "recall_neuropil"     : float(recall[NEU_ID]),
        "recall_soma"         : float(recall[SOMA_ID]),
        "confusion_matrix"    : cm.tolist(),
    }


# Run on all VAL images using CHECKPOINT_TO_LOAD
val_results = []
for image_id in VAL_IDS:
    print(f"Running full-slide inference: {image_id} with {CHECKPOINT_TO_LOAD}")
    r = run_full_slide_with_gt(
        image_id        = image_id,
        model           = model,
        device          = device,
        path_normalized = PATH_NORMALIZED,
        path_masks      = PATH_MASKS,
        pad_lookup      = all_pad_lookup,
        out_dir         = os.path.join(OUT_DIR, "val_fullslide", CHECKPOINT_TO_LOAD.replace(".pt", "")),
        patch_size      = PATCH_SIZE,
        stride          = STRIDE,
        infer_batch_size= 4,
    )
    print(f"  Dice      bg={r['dice_bg']:.3f} neu={r['dice_neuropil']:.3f} soma={r['dice_soma']:.3f}")
    print(f"  Precision bg={r['precision_bg']:.3f} neu={r['precision_neuropil']:.3f} soma={r['precision_soma']:.3f}")
    print(f"  Recall    bg={r['recall_bg']:.3f} neu={r['recall_neuropil']:.3f} soma={r['recall_soma']:.3f}")
    val_results.append(r)

val_results_df = pd.DataFrame(val_results)
val_results_df.to_csv(os.path.join(PATH_SPLITS, f"val_fullslide_metrics_{CHECKPOINT_TO_LOAD.replace('.pt','')}.csv"), index=False)
print("\nSaved full-slide validation metrics CSV")


## 20. Qualitative Full-Slide Inference on Unlabeled Test Images

No ground-truth masks exist for these slides, so no quantitative metrics
are computed. We save the prediction RGB and overlay for visual inspection.
This is the primary output for test-set assessment at the pilot stage.

In [ ]:
def run_full_slide_no_gt(
    image_id, model, device,
    path_normalized, pad_lookup, out_dir,
    patch_size=512, stride=256, infer_batch_size=4,
):
    """
    Run full-slide inference on an UNLABELED image — qualitative output only.
    """
    img_bgr = cv2.imread(os.path.join(path_normalized, f"{image_id}.png"), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f"Test image not found: {image_id}.png in {path_normalized}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if image_id in pad_lookup:
        pad = pad_lookup[image_id]
    else:
        h, w = img_rgb.shape[:2]
        pad, _ = compute_pad_to_multiple(h, w, multiple=patch_size)

    pred_ids = predict_full_slide(model, img_rgb, pad, device,
                                  patch_size=patch_size, stride=stride,
                                  infer_batch_size=infer_batch_size)

    os.makedirs(out_dir, exist_ok=True)
    pred_rgb = ids_to_rgb(pred_ids)
    overlay  = overlay_pred_on_image(img_rgb, pred_ids)

    cv2.imwrite(os.path.join(out_dir, f"{image_id}_pred_ids.png"), pred_ids)
    cv2.imwrite(os.path.join(out_dir, f"{image_id}_pred_rgb.png"), cv2.cvtColor(pred_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(out_dir, f"{image_id}_overlay.png"),  cv2.cvtColor(overlay,  cv2.COLOR_RGB2BGR))

    print(f"  {image_id}: saved pred_rgb and overlay to {out_dir}")


# Run qualitative inference on all unlabeled test images
print(f"=== Qualitative inference on unlabeled test images using {CHECKPOINT_TO_LOAD} ===")
test_output_dir = os.path.join(OUT_DIR, "test_qualitative", CHECKPOINT_TO_LOAD.replace(".pt", ""))
for image_id in TEST_IDS:
    run_full_slide_no_gt(
        image_id        = image_id,
        model           = model,
        device          = device,
        path_normalized = PATH_NORMALIZED,
        pad_lookup      = all_pad_lookup,
        out_dir         = test_output_dir,
        patch_size      = PATCH_SIZE,
        stride          = STRIDE,
        infer_batch_size= 4,
    )
print("Done. Inspect the test_qualitative/ folder for overlays.")


## 21. Inline Preview of Test Slide Predictions

Display a random crop from each test slide's overlay directly in the notebook
for a quick sanity check without opening the full multi-gigapixel PNG.

In [ ]:
def preview_test_output(image_id, out_dir, crop_size=2048):
    """Load and display a random crop of the test overlay image."""
    overlay_path = os.path.join(out_dir, f"{image_id}_overlay.png")
    overlay_bgr  = cv2.imread(overlay_path, cv2.IMREAD_COLOR)
    if overlay_bgr is None:
        print(f"Overlay not found for {image_id} — run inference first.")
        return

    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
    h, w = overlay_rgb.shape[:2]

    max_y = max(h - crop_size, 0)
    max_x = max(w - crop_size, 0)
    y0    = random.randint(0, max_y)
    x0    = random.randint(0, max_x)
    crop  = overlay_rgb[y0:y0 + crop_size, x0:x0 + crop_size]

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(crop)
    ax.set_title(f"{image_id}  ({CHECKPOINT_TO_LOAD}, random {crop_size}px crop)")
    ax.axis("off")
    fig.legend(handles=class_legend(), loc="lower center", ncol=3, fontsize=10)
    plt.tight_layout()
    plt.show()


for image_id in TEST_IDS:
    preview_test_output(image_id, test_output_dir, crop_size=2048)
